In [ ]:
# 1-2. Wczytanie danych (sciezki niezalezne od komputera)
from pathlib import Path
import json
import requests
import geopandas as gpd
import pandas as pd

NAZWA_TRZESIENIA = "trzesienia_2025.geojson"
NAZWA_PANSTWA = "ne_110m_admin_0_countries.zip"

# Szukamy folderu z danymi wzgledem miejsca, w ktorym jest notatnik:
# najpierw w biezacym katalogu, potem w katalogach nadrzednych (az do glownego folderu repo).
# Dzieki temu notatnik dziala u kazdego, niezaleznie od tego, gdzie sklonowal repozytorium.
TUTAJ = Path.cwd()
DANE = None
for katalog in [TUTAJ, *TUTAJ.parents]:
    for kandydat in [katalog, katalog / "6", katalog / "data"]:
        if (kandydat / NAZWA_TRZESIENIA).exists():
            DANE = kandydat
            break
    if DANE:
        break

if DANE is None:
    # nie znaleziono danych -> tworzymy folder 6/ obok notatnika i pobierzemy dane z internetu
    DANE = TUTAJ / "6"
    DANE.mkdir(exist_ok=True)

print("Folder z danymi:", DANE)

SCIEZKA_TRZESIENIA = DANE / NAZWA_TRZESIENIA
SCIEZKA_PANSTWA = DANE / NAZWA_PANSTWA

# --- Trzesienia ziemi (USGS) ---
url = "https://earthquake.usgs.gov/fdsnws/event/1/query"

# okres 1 roku, Europa i morze srodziemne, m>=3
parametry = {
    "format": "geojson",
    "starttime": "2025-01-01",
    "endtime": "2026-01-01",
    "minmagnitude": 3.0,
    "minlatitude": 30.0,   # poludnie (M. Śródziemne)
    "maxlatitude": 72.0,   # polnoc (Skandynawia)
    "minlongitude": -25.0, # zachod (Islandia)
    "maxlongitude": 45.0   # wschod (Kaukaz)
}

if SCIEZKA_TRZESIENIA.exists():
    # plik jest w repo -> wczytujemy go, bez laczenia z USGS
    print("Wczytuję dane z pliku:", SCIEZKA_TRZESIENIA)
    dane_json = json.loads(SCIEZKA_TRZESIENIA.read_text(encoding="utf-8"))
else:
    # nie ma pliku -> pobieramy z USGS i zapisujemy
    print("Brak pliku, łączę z USGS...")
    odpowiedz = requests.get(url, params=parametry, timeout=60)
    odpowiedz.raise_for_status()   # jesli USGS zwroci blad, od razu to zobaczymy
    dane_json = odpowiedz.json()
    SCIEZKA_TRZESIENIA.write_text(json.dumps(dane_json), encoding="utf-8")
    print("Zapisano:", SCIEZKA_TRZESIENIA)

gdf = gpd.GeoDataFrame.from_features(dane_json["features"], crs="EPSG:4326")  # WGS 84

# --- Granice panstw (Natural Earth) ---
if not SCIEZKA_PANSTWA.exists():
    print("Brak granic państw, pobieram z Natural Earth...")
    zip_url = "https://naciscdn.org/naturalearth/110m/cultural/ne_110m_admin_0_countries.zip"
    odp = requests.get(zip_url, timeout=60, headers={"User-Agent": "Mozilla/5.0"})
    odp.raise_for_status()
    SCIEZKA_PANSTWA.write_bytes(odp.content)
    print("Zapisano:", SCIEZKA_PANSTWA)

print("Pobrano wierszy:", len(gdf))
print(gdf.head(3)) # pierwsze 3 trzesienia

In [2]:
# wspolrzedna z, szerokosc i dlugosc geo
gdf["depth_km"] = gdf.geometry.z
gdf["longitude"] = gdf.geometry.x
gdf["latitude"] = gdf.geometry.y

In [8]:
print(f"liczba zdarzen: {len(gdf)}")
print("\nuklad wspolrzednych (CRS):", gdf.crs)
print("\ntypy geometrii:\n", gdf.geom_type.value_counts())
print("\npola magnitudy, glebokosci i czasu:", gdf[["mag","depth_km","time"]].head())
print("\nstatystyki opisowe:\n", gdf[["mag","depth_km"]].describe())
print("\nbraki danych", gdf[["mag","depth_km","time"]].isna().sum())

liczba zdarzen: 840

uklad wspolrzednych (CRS): EPSG:4326

typy geometrii:
 Point    840
Name: count, dtype: int64

pola magnitudy, glebokosci i czasu:    mag  depth_km           time
0  4.4    10.000  1767107131518
1  4.2    44.648  1767027606759
2  4.1    10.000  1766616206824
3  4.2   146.758  1766437760250
4  4.0    10.000  1766272436133

statystyki opisowe:
               mag    depth_km
count  840.000000  840.000000
mean     4.343690   17.717938
std      0.367685   27.603639
min      3.000000    0.000000
25%      4.100000   10.000000
50%      4.300000   10.000000
75%      4.500000   10.000000
max      6.500000  404.050000

braki danych mag         0
depth_km    0
time        0
dtype: int64


In [ ]:
#4
gdf["datetime"] = pd.to_datetime(gdf["time"], unit="ms", utc=True)

gdf_czyste =gdf[
    gdf["geometry"].notna() &
    gdf.geometry.is_valid &
    (~gdf["geometry"].is_empty) &
    gdf["mag"].notna()
].copy()

print(f"poprawne zdarzenia: {len(gdf_czyste)}")
print(gdf_czyste[["datetime","mag","depth_km","place"]].head())

In [ ]:
# 5
import matplotlib.pyplot as plt

# sciezka SCIEZKA_PANSTWA ustawiona w pierwszej komorce (folder data/ w repo)
gdf_panstwa = gpd.read_file(SCIEZKA_PANSTWA)

if gdf_panstwa.crs != gdf_czyste.crs:
    gdf_panstwa = gdf_panstwa.to_crs(gdf_czyste.crs)

fig, ax = plt.subplots(figsize=(12, 9))
ax.set_facecolor("#e6f2ff")

gdf_panstwa.plot(
    ax=ax,
    color="#f2f2f2",
    edgecolor="#7f7f7f",
    linewidth=0.6
)


gdf_czyste.plot(
    ax=ax,
    column="mag",
    cmap="YlOrRd",
    markersize=gdf_czyste["mag"] ** 2.3,
    alpha=0.75,
    edgecolor="black",
    linewidth=0.3,
    legend=True,
    legend_kwds={"label": "Magnituda (M)", "shrink": 0.6}
)

ax.set_xlim(-25, 45)
ax.set_ylim(30, 72)

ax.set_title("Rozmieszczenie epicentrów trzęsień ziemi (M ≥ 3,0) w Europie i basenie M. Śródziemnego", fontsize=13, pad=12)
ax.set_xlabel("Długość geograficzna [°]")
ax.set_ylabel("Szerokość geograficzna [°]")
ax.grid(True, linestyle="--", alpha=0.4, color="gray")

plt.tight_layout()
plt.show()


In [ ]:
# 6. Przypisanie epicentrow na ladzie do panstw (sjoin), zdarzenia morskie osobno
gdf_sj = gpd.sjoin(
    gdf_czyste,
    gdf_panstwa[["NAME", "geometry"]],
    how="left",          # left = zostaw tez punkty, ktore nie trafily w zadne panstwo
    predicate="within"
)

# punkt na granicy dwoch panstw moglby sie zdublowac - zostawiamy jeden
gdf_sj = gdf_sj[~gdf_sj.index.duplicated(keep="first")]

# brak panstwa = zdarzenie na morzu
gdf_sj["panstwo"] = gdf_sj["NAME"].fillna("Morze")
gdf_sj = gdf_sj.drop(columns=["index_right", "NAME"])

print("Na lądzie:", (gdf_sj["panstwo"] != "Morze").sum())
print("Na morzu:", (gdf_sj["panstwo"] == "Morze").sum())

# Uwaga: warstwa 110m ma uproszczone granice, wiec czesc zdarzen
# tuz przy brzegu moze zostac zakwalifikowana jako "Morze".

In [ ]:
# 7. Liczba zdarzen w panstwach i 10 obszarow z najwieksza liczba zdarzen
liczba_zdarzen = gdf_sj["panstwo"].value_counts()
print("Liczba zdarzeń w poszczególnych obszarach:")
print(liczba_zdarzen)

top10 = liczba_zdarzen.head(10)
print("\nTOP 10 obszarów:")
print(top10)

fig, ax = plt.subplots(figsize=(10, 6))
top10.sort_values().plot(kind="barh", ax=ax, color="#d95f0e")
ax.set_title("10 obszarów z największą liczbą trzęsień ziemi (M ≥ 3,0)")
ax.set_xlabel("Liczba zdarzeń")
ax.set_ylabel("")
plt.tight_layout()
plt.show()

In [ ]:
# 8. Porownanie rozmieszczenia epicentrow M >= 3,0 i M >= 5,0
m3 = gdf_sj[gdf_sj["mag"] >= 3.0]
m5 = gdf_sj[gdf_sj["mag"] >= 5.0]

print(f"Zdarzenia M >= 3,0: {len(m3)}")
print(f"Zdarzenia M >= 5,0: {len(m5)} ({len(m5) / len(m3) * 100:.1f}% wszystkich)")

print("\nNajwięcej zdarzeń M >= 5,0:")
print(m5["panstwo"].value_counts().head(10))

# dwie mapy obok siebie
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
for ax, dane, tytul, kolor in [
    (axes[0], m3, "M ≥ 3,0", "orange"),
    (axes[1], m5, "M ≥ 5,0", "red"),
]:
    ax.set_facecolor("#e6f2ff")
    gdf_panstwa.plot(ax=ax, color="#f2f2f2", edgecolor="#7f7f7f", linewidth=0.6)
    dane.plot(ax=ax, color=kolor, markersize=15, alpha=0.7, edgecolor="black", linewidth=0.3)
    ax.set_xlim(-25, 45)
    ax.set_ylim(30, 72)
    ax.set_title(f"Epicentra {tytul} (n = {len(dane)})")
    ax.grid(True, linestyle="--", alpha=0.4)

plt.tight_layout()
plt.show()

# Wniosek: slabsze trzesienia (M >= 3) wystepuja w calym pasie od Islandii,
# przez Wlochy i Balkany, po Turcje. Silniejsze (M >= 5) skupiaja sie
# glownie w Grecji, Turcji i na Morzu Egejskim - tam gdzie stykaja sie plyty tektoniczne.

In [ ]:
# 9. Srednia, mediana i maksimum magnitudy oraz glebokosci
statystyki = gdf_sj[["mag", "depth_km"]].agg(["mean", "median", "max"]).round(2)
statystyki.index = ["średnia", "mediana", "maksimum"]
statystyki.columns = ["magnituda [M]", "głębokość [km]"]
print(statystyki)

# Uwaga: mediana glebokosci = 10 km, bo USGS dla wielu slabszych zdarzen
# przyjmuje glebokosc "na sztywno" 10 km, gdy nie da sie jej dokladnie wyznaczyc.
print("\nZdarzenia z głębokością dokładnie 10 km:", (gdf_sj["depth_km"] == 10).sum())

In [ ]:
# 10. Podzial na trzy grupy glebokosci
# klasyczny podzial sejsmologiczny:
#   plytkie   0 - 70 km
#   posrednie 70 - 300 km
#   glebokie  > 300 km
przedzialy = [0, 70, 300, float("inf")]
etykiety = ["płytkie (0–70 km)", "pośrednie (70–300 km)", "głębokie (>300 km)"]

gdf_sj["grupa_glebokosci"] = pd.cut(
    gdf_sj["depth_km"],
    bins=przedzialy,
    labels=etykiety,
    include_lowest=True  # zeby glebokosc 0 km tez wpadla do "plytkich"
)

grupy = gdf_sj.groupby("grupa_glebokosci", observed=False).agg(
    liczba=("mag", "size"),
    srednia_mag=("mag", "mean"),
    max_mag=("mag", "max")
)
grupy["udzial_%"] = (grupy["liczba"] / grupy["liczba"].sum() * 100).round(1)
print(grupy.round(2))

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(grupy.index.astype(str), grupy["liczba"], color=["#fdae6b", "#e6550d", "#7f2704"])
for i, v in enumerate(grupy["liczba"]):
    ax.text(i, v, str(v), ha="center", va="bottom")
ax.set_title("Liczba trzęsień ziemi w grupach głębokości")
ax.set_ylabel("Liczba zdarzeń")
plt.tight_layout()
plt.show()

# Wniosek: zdecydowana wiekszosc trzesien to zdarzenia plytkie (skorupa ziemska).
# Posrednie i glebokie wystepuja glownie tam, gdzie plyta afrykanska wchodzi pod euroazjatycka
# (luk Helenski - Grecja/Kreta, Morze Tyrrenskie - poludniowe Wlochy).